In [ ]:
# Setup Dash for local browser or JupyterLab external mode

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, Dash
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import the CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# initalize the client (uses defaults)
shelter = AnimalShelter()

username = "aacuser"
password = "SNHU1234"

# log the user in to the database
shelter.login(username, password)


# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
# Dash app used for both local browser and JupyterLab external mode
app = Dash(__name__)

header_image_filename = 'Grazioso Salvare Logo.png'
header_image = base64.b64encode(open(header_image_filename, 'rb').read())
header_image_html = html.Img(src=f'data:image/png;base64,{header_image.decode()}', height=200, width=200)
dropdown_options = [
            {'label': 'Water Rescue', 'value': 1 },
            {'label': 'Mountain or Wilderness Rescue', 'value': 2 },
            {'label': 'Disaster Rescue or Individual Tracking', 'value': 3 }
        ]


app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H2('Emilio Crocco - 2/17/2026')),
    html.Hr(),
    html.Center(html.A(children=[header_image_html], href='https://www.snhu.edu')),
    html.Hr(),
    html.Div(children=[
        html.Label('Select a filter and press "X" to clear'),
        dcc.Dropdown(
            id='filter-type',
            options=dropdown_options,
            value=None,
            searchable=False,
            placeholder="Select a filter",
            style={'max-width': '30%'}
        ),
    ]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        editable=False,
        filter_action='native',
        sort_action='native',
        sort_mode='multi',
        column_selectable=False,
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action='native',
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex', 'gap': '2%'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'flex': '1', 'minWidth': '0'}
            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'flex': '1', 'minWidth': '0'}
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(Output('datatable-id', 'data'),
              Output('datatable-id', 'page_current'),
              Output('datatable-id', 'selected_rows'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    water_rescue_query = { 'animal_type': 'Dog', 'breed': { '$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']}, 'sex_upon_outcome': 'Intact Female', 'age_upon_outcome_in_weeks': { '$gte': 26, '$lte': 156 }}
    mountain_wilderness_query = { 'animal_type': 'Dog', 'breed': { '$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky','Rottweiler']}, 'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': { '$gte': 26, '$lte': 156 }}
    disaster_individual_query = { 'animal_type': 'Dog', 'breed': { '$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever','Bloodhound', 'Rottweiler']}, 'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': { '$gte': 20, '$lte': 300 }}

    if filter_type == 1:
        dff = pd.DataFrame.from_records(shelter.read(water_rescue_query))
        dff.drop(columns=['_id'], inplace=True, errors='ignore')
    elif filter_type == 2:
        dff = pd.DataFrame.from_records(shelter.read(mountain_wilderness_query))
        dff.drop(columns=['_id'], inplace=True, errors='ignore')
    elif filter_type == 3:
        dff = pd.DataFrame.from_records(shelter.read(disaster_individual_query))
        dff.drop(columns=['_id'], inplace=True, errors='ignore')
    else:
        dff = df

    data = dff.to_dict('records')
    return data, 0, [0]


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if not viewData:
        chart_data = df
    else:
        chart_data = pd.DataFrame.from_dict(viewData)

    # Count breeds, then group anything under 1% into 'Other'
    breed_counts = chart_data['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    total = breed_counts['count'].sum()
    breed_counts['breed'] = breed_counts.apply(
        lambda row: row['breed'] if (row['count'] / total * 100) >= 0.75 else 'Other',
        axis=1
    )
    breed_counts = breed_counts.groupby('breed', as_index=False)['count'].sum()

    return [
        dcc.Graph(
            figure=px.pie(breed_counts, names='breed', values='count', title='Preferred Animals')
        )
    ]


#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Adding a clean map that doesnt rely on viewData
    clean_map = dl.Map(style={'width': '1000px', 'height': '500px'},
                    center=[30.75,-97.48], zoom=10, children=[
                    dl.TileLayer(id="base-layer-id")
                ])

    # Return the map if no data is present to prevent accessing viewData early
    if viewData is None or len(viewData) == 0:
        return [
            html.Div(id='container-div', style={'display':'flex'},
            children=[clean_map])
        ]

    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can
    # be converted to a row index here
    if not index:
        row = 0
    else:
        row = index[0]

    # Return a clean map with no marker if no grid-coordinates are present
    if pd.isna(pd.to_numeric(dff.iloc[row,13], errors='coerce')) or pd.isna(pd.to_numeric(dff.iloc[row,14], errors='coerce')):
        return [
            html.Div(id='container-div', style={'display':'flex'},
            children=[clean_map])
        ]

    return [
        html.Div(id='container-div', style={'display':'flex'},
            children=[
                dl.Map(style={'width': '1000px', 'height': '500px'},
                    center=[dff.iloc[row,13],dff.iloc[row,14]], zoom=10, children=[
                    dl.TileLayer(id="base-layer-id"),
                    # Marker with tool tip and popup
                    # Column 13 and 14 define the grid-coordinates for the map
                    # Column 4 defines the breed for the animal
                    # Column 9 defines the name of the animal
                    dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
                        children=[
                            dl.Tooltip(dff.iloc[row,4]),
                            dl.Popup([
                                html.H1("Animal Name"),
                                html.P(dff.iloc[row,9])
                        ])
                    ]),
                ]),
            ]
        ),
    ]


# Run the app in a browser. In JupyterLab, Dash prints an external URL.
app.run(jupyter_mode='external')